In [1]:
import pandas as pd
import folium


# ============================================================
# 1. Función para convertir coordenadas NMEA a grados decimales
# ============================================================

def nmea_to_decimal(coord, hemisphere):
    """
    Convierte coordenadas NMEA:
        lat: ddmm.mmmm
        lon: dddmm.mmmm

    Ejemplo:
        1205.9943 S -> -12.099905
        07701.4552 W -> -77.024253
    """

    if coord == "" or pd.isna(coord):
        return None

    value = float(coord)

    degrees = int(value // 100)
    minutes = value - degrees * 100

    decimal = degrees + minutes / 60.0

    if hemisphere in ["S", "W"]:
        decimal *= -1

    return decimal


# ============================================================
# 2. Leer CSV reconstruyendo la sentencia GPS_data
# ============================================================

def read_gps_csv(filepath):
    rows = []

    with open(filepath, "r", encoding="utf-8") as f:
        header = next(f)  # saltar encabezado

        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split(",")

            unix_time = parts[0]
            motion_flag = parts[-1]

            # Reconstruir toda la sentencia NMEA
            nmea = ",".join(parts[1:-1])

            rows.append({
                "Unix_time": unix_time,
                "GPS_data": nmea,
                "motion_flag": motion_flag
            })

    return pd.DataFrame(rows)


# ============================================================
# 3. Parsear sentencias GPGGA y GPRMC
# ============================================================

def parse_nmea_dataframe(df):
    parsed_rows = []

    for _, row in df.iterrows():
        nmea = row["GPS_data"]
        fields = nmea.split(",")

        sentence_type = fields[0]

        lat = None
        lon = None
        altitude = None
        satellites = None
        hdop = None
        speed_knots = None
        course_deg = None
        gps_time = None
        gps_date = None

        # -----------------------------
        # GPGGA: posición, altura, satélites
        # -----------------------------
        if sentence_type == "$GPGGA":
            gps_time = fields[1]

            lat = nmea_to_decimal(fields[2], fields[3])
            lon = nmea_to_decimal(fields[4], fields[5])

            fix_quality = fields[6]
            satellites = fields[7]
            hdop = fields[8]
            altitude = fields[9]

            parsed_rows.append({
                "Unix_time": row["Unix_time"],
                "sentence": sentence_type,
                "gps_time": gps_time,
                "gps_date": gps_date,
                "lat": lat,
                "lon": lon,
                "altitude_m": float(altitude) if altitude != "" else None,
                "satellites": int(satellites) if satellites != "" else None,
                "hdop": float(hdop) if hdop != "" else None,
                "speed_knots": speed_knots,
                "speed_kmh": None,
                "course_deg": course_deg,
                "motion_flag": int(row["motion_flag"])
            })

        # -----------------------------
        # GPRMC: posición, velocidad, rumbo, fecha
        # -----------------------------
        elif sentence_type == "$GPRMC":
            gps_time = fields[1]
            status = fields[2]

            # A = válido, V = inválido
            if status != "A":
                continue

            lat = nmea_to_decimal(fields[3], fields[4])
            lon = nmea_to_decimal(fields[5], fields[6])

            speed_knots = fields[7]
            course_deg = fields[8]
            gps_date = fields[9]

            speed_kmh = float(speed_knots) * 1.852 if speed_knots != "" else None

            parsed_rows.append({
                "Unix_time": row["Unix_time"],
                "sentence": sentence_type,
                "gps_time": gps_time,
                "gps_date": gps_date,
                "lat": lat,
                "lon": lon,
                "altitude_m": None,
                "satellites": None,
                "hdop": None,
                "speed_knots": float(speed_knots) if speed_knots != "" else None,
                "speed_kmh": speed_kmh,
                "course_deg": float(course_deg) if course_deg != "" else None,
                "motion_flag": int(row["motion_flag"])
            })

    parsed = pd.DataFrame(parsed_rows)

    parsed = parsed.dropna(subset=["lat", "lon"]).reset_index(drop=True)

    return parsed


# ============================================================
# 4. Crear mapa interactivo
# ============================================================

def plot_gps_on_map(df_gps, output_html="gps_track_map.html"):
    if df_gps.empty:
        raise ValueError("No hay datos GPS válidos para plotear.")

    center_lat = df_gps["lat"].mean()
    center_lon = df_gps["lon"].mean()

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=18,
        tiles="OpenStreetMap"
    )

    coords = df_gps[["lat", "lon"]].values.tolist()

    # Línea del recorrido
    folium.PolyLine(
        coords,
        weight=5,
        opacity=0.8,
        tooltip="Trayectoria GPS"
    ).add_to(m)

    # Punto inicial
    folium.Marker(
        coords[0],
        popup="Inicio",
        tooltip="Inicio",
        icon=folium.Icon(color="green", icon="play")
    ).add_to(m)

    # Punto final
    folium.Marker(
        coords[-1],
        popup="Fin",
        tooltip="Fin",
        icon=folium.Icon(color="red", icon="stop")
    ).add_to(m)

    # Puntos intermedios
    for i, row in df_gps.iterrows():
        popup = f"""
        Índice: {i}<br>
        Lat: {row['lat']}<br>
        Lon: {row['lon']}<br>
        Sentencia: {row['sentence']}<br>
        Motion flag: {row['motion_flag']}<br>
        Velocidad km/h: {row['speed_kmh']}
        """

        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=3,
            popup=popup,
            fill=True,
            fill_opacity=0.8
        ).add_to(m)

    m.save(output_html)

    print(f"Mapa guardado en: {output_html}")


# ============================================================
# 5. Uso
# ============================================================

filepath = "GPSlog__1.csv"

df_raw = read_gps_csv(filepath)
df_gps = parse_nmea_dataframe(df_raw)

print(df_gps.head())
print(df_gps[["lat", "lon"]].describe())

plot_gps_on_map(df_gps, output_html="gps_track_map.html")

       Unix_time sentence    gps_time gps_date        lat        lon  \
0  1779389613248   $GPRMC  185122.000   210526 -12.099897 -77.024262   
1  1779389613248   $GPGGA  185123.000      NaN -12.099897 -77.024262   
2  1779389613252   $GPRMC  185123.000   210526 -12.099897 -77.024262   
3  1779389613252   $GPGGA  185124.000      NaN -12.099897 -77.024262   
4  1779389613253   $GPRMC  185124.000   210526 -12.099897 -77.024262   

   altitude_m  satellites  hdop  speed_knots  speed_kmh  course_deg  \
0         NaN         NaN   NaN         0.16    0.29632      294.38   
1       142.9         7.0   1.3          NaN        NaN         NaN   
2         NaN         NaN   NaN         0.02    0.03704      294.38   
3       142.8         7.0   1.3          NaN        NaN         NaN   
4         NaN         NaN   NaN         0.01    0.01852      294.38   

   motion_flag  
0            0  
1            0  
2            0  
3            0  
4            0  
              lat         lon
count  9

In [3]:
import folium

m = folium.Map(
    location=[df_gps["lat"].mean(), df_gps["lon"].mean()],
    zoom_start=18
)

m